# C1: Text Summarization using Transformer Models

Use a pre-trained model (BART / T5) to summarise long news articles, evaluate the summaries with ROUGE-1, ROUGE-2 and ROUGE-L, and compare extractive and abstractive summarisation.

**Dataset:** CNN/Daily Mail (Hugging Face hub) and the Kaggle News Summary dataset (Indian news from Inshorts, with 60-word human summaries)

**Runtime -> Change runtime type -> T4 GPU** before running. Cells that download BART, T5 or CNN/Daily Mail have no saved output in this file: run them in Colab.

In [2]:
# transformers  : BART and T5
# datasets      : CNN/Daily Mail
# sentencepiece : the tokenizer T5 needs
# rouge-score   : ROUGE
# networkx      : PageRank for TextRank
!pip install -q transformers datasets sentencepiece rouge-score nltk scikit-learn networkx pandas

In [3]:
import re
import numpy as np, pandas as pd
import nltk
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer

## Data: news article dataset

In [4]:
news = pd.read_csv("https://raw.githubusercontent.com/sunnysai12345/News_Summary/master/news_summary.csv", encoding="latin-1")
news = news.dropna(subset=["ctext", "text"])
news[["headlines", "text", "ctext"]].head(2)

,headlines,text,ctext
0,Daman & Diu revokes mandatory Rakshabandhan in...,The Administration of Union Territory Daman an...,The Daman and Diu administration on Wednesday ...
1,Malaika slams user who trolled her for 'divorc...,Malaika Arora slammed an Instagram user who tr...,"From her special numbers to TV?appearances, Bo..."


In [5]:
def clean(t):
    t = re.sub(r"(?<=\w)\?(?=[st]\b)", "'", t)       # territory?s -> territory's (quotes were lost in the file)
    t = re.sub(r"(^|\s)\?(?=\w)", r'\1"', t)              # ?free -> "free
    t = re.sub(r"(?<=\w)\?(?=[,.;:]|\s+[a-z])", '"', t)     # passage? for -> passage" for
    t = re.sub(r"(?<=[.,])\?", '"', t)                         # Shillong.? -> Shillong."
    t = re.sub(r"([.!?])([A-Z])", r"\1 \2", t)        # media.The -> media. The
    return re.sub(r"\s+", " ", t).strip()

news["article"] = news["ctext"].map(clean)
news["summary"] = news["text"].map(clean)
news = news[news["article"].str.split().str.len().between(150, 800)].sample(50, random_state=9).reset_index(drop=True)
print(len(news), "articles,", round(news["article"].str.split().str.len().mean()), "words on average")
news["summary"][0]

50 articles, 335 words on average


"Australia's Queensland is building a 2,000-kilometre network of electric vehicle charging stations, which will make up one of the longest electric vehicle highways in the world. The Electric Super Highway will be completed within six months. Its stations will offer free power for at least one year, in an attempt to boost the number of electric cars in Queensland."

## Data: CNN / Daily Mail

In [ ]:
from datasets import load_dataset
cnn = load_dataset("abisee/cnn_dailymail", "3.0.0", split="test[:50]")
cnn = pd.DataFrame({"article": cnn["article"], "summary": cnn["highlights"]})
cnn["summary"][0]

## Extractive summaries

- **Lead-3:** the first three sentences (news puts the key facts first).
- **TextRank:** sentences are nodes, TF-IDF similarity between sentences are edges, PageRank picks the most central sentences.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx

def lead3(article):
    return " ".join(sent_tokenize(article)[:3])

def textrank(article, k=3):
    sents = sent_tokenize(article)
    if len(sents) <= k:
        return article
    X = TfidfVectorizer(stop_words="english").fit_transform(sents)
    sim = (X @ X.T).toarray()                       # similarity between every pair of sentences
    scores = nx.pagerank(nx.from_numpy_array(sim))
    top = sorted(sorted(scores, key=scores.get, reverse=True)[:k])    # best k, in article order
    return " ".join(sents[i] for i in top)

print("LEAD-3:  ", lead3(news["article"][0])); print()
print("TEXTRANK:", textrank(news["article"][0]))

LEAD-3:   Queensland will have a 2,000km network of electric vehicle charging stations that make up one of the world's longest electric vehicle highways within six months. The state government announced on Thursday it would build an 18-station network stretching along Queensland's east coast from Cairns to Coolangatta and west to Toowoomba. The stations, which recharge a vehicle in 30 minutes, will offer free power for at least a year in what the environment minister, Steven Miles, said was a bid to boost the number of electric cars on Queensland roads, currently about 700.

TEXTRANK: Queensland will have a 2,000km network of electric vehicle charging stations that make up one of the world's longest electric vehicle highways within six months. Miles said a state household energy survey showed half of Queenslanders would consider "an electric vehicle, plug-in hybrid or regenerative braking hybrid" when buying a new car in the next two years. "Behyad Jafari, the chief executive of the El

## ROUGE-1, ROUGE-2, ROUGE-L

In [8]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
print(scorer.score("India won the final by six wickets", "India beat Australia in the final"))

{'rouge1': Score(precision=0.5, recall=0.42857142857142855, fmeasure=0.4615384615384615), 'rouge2': Score(precision=0.2, recall=0.16666666666666666, fmeasure=0.1818181818181818), 'rougeL': Score(precision=0.5, recall=0.42857142857142855, fmeasure=0.4615384615384615)}


In [9]:
def rouge(summaries, references):
    scores = [scorer.score(r, s) for s, r in zip(summaries, references)]
    return {m: round(100 * np.mean([s[m].fmeasure for s in scores]), 1) for m in ["rouge1", "rouge2", "rougeL"]}

news_results = {"Lead-3": rouge([lead3(a) for a in news["article"]], news["summary"]),
                "TextRank": rouge([textrank(a) for a in news["article"]], news["summary"])}
pd.DataFrame(news_results).T

,rouge1,rouge2,rougeL
Lead-3,43.5,21.8,30.9
TextRank,41.7,18.9,28.0


## (a, b) Abstractive summaries with BART and T5

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

models = {}
for name in ["facebook/bart-large-cnn", "t5-small"]:
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)
    models[name] = (tok, model)
    print(name, "- input limit:", tok.model_max_length, "tokens")

In [ ]:
def summarize(name, articles, batch_size=8):
    tok, model = models[name]
    prefix = "summarize: " if "t5" in name else ""          # T5 needs the task written in front of the text
    summaries = []
    for i in range(0, len(articles), batch_size):
        batch = [prefix + a for a in articles[i:i + batch_size]]
        inputs = tok(batch, truncation=True, max_length=tok.model_max_length, padding=True, return_tensors="pt").to(device)
        ids = model.generate(**inputs, num_beams=4, max_new_tokens=80, min_new_tokens=30, no_repeat_ngram_size=3)
        summaries += tok.batch_decode(ids, skip_special_tokens=True)
    return summaries

for name in models:
    print(name, ":", summarize(name, [news["article"][0]])[0]); print()
print("HUMAN:", news["summary"][0])

In [ ]:
news_out = {name: summarize(name, list(news["article"])) for name in models}
cnn_out = {name: summarize(name, list(cnn["article"])) for name in models}

## (c) ROUGE for all systems

In [ ]:
cnn_results = {"Lead-3": rouge([lead3(a) for a in cnn["article"]], cnn["summary"]),
               "TextRank": rouge([textrank(a) for a in cnn["article"]], cnn["summary"])}
for name in models:
    news_results[name] = rouge(news_out[name], news["summary"])
    cnn_results[name] = rouge(cnn_out[name], cnn["summary"])

pd.concat({"News Summary": pd.DataFrame(news_results).T, "CNN/Daily Mail": pd.DataFrame(cnn_results).T})

## (d) Extractive vs abstractive

In [ ]:
def new_words(summary, article):
    s, a = set(re.findall(r"[a-z]+", summary.lower())), set(re.findall(r"[a-z]+", article.lower()))
    return len(s - a) / max(1, len(s))           # share of summary words not in the article

rows = []
for name, outs in [("Lead-3", [lead3(a) for a in news["article"]]), ("TextRank", [textrank(a) for a in news["article"]])] + list(news_out.items()):
    rows.append({"system": name, "words": np.mean([len(o.split()) for o in outs]),
                 "new words %": 100 * np.mean([new_words(o, a) for o, a in zip(outs, news["article"])])})
pd.DataFrame(rows).set_index("system").round(1)

In [ ]:
i = 3
print("HUMAN:   ", news["summary"][i]); print()
print("LEAD-3:  ", lead3(news["article"][i])); print()
for name in models:
    print(name, ":", news_out[name][i]); print()

## Conclusion
- Extractive summaries (Lead-3, TextRank) copy sentences: always faithful to the article, but long and choppy, and they cannot rephrase.
- Abstractive summaries (BART, T5) are shorter and fluent and use new words, but can state things the article does not say: check names and numbers by hand.
- Compare ROUGE only on the same dataset. BART, fine-tuned on CNN/Daily Mail, is expected to do best there; on the Inshorts news the styles differ.
- BART reads at most 1,024 tokens and T5 512; longer articles are cut (or split into chunks and summarised piece by piece).